# 02: Clean and prepare TriMet monthly data

**Goal:** Produce a table of monthly TriMet UPT and VRH data from January 2019 to September 2026, ready for SQL analysis.

**Input:** `data/raw/ntd_monthly_complete_2026-11-15.xlsx` (not committed; see the README for download instructions)

**Output:** `data/processed/trimet_monthly.csv`, one row per mode, type of service, and month.

**Steps:** Load UPT and VRH, reshape, merge, trim, check core modes' share of total UPT, validate, and save.

## Set up and loader

In [1]:
import re
import pandas as pd

PATH = "data/raw/ntd_monthly_complete_2026-11-15.xlsx"
OUT_PATH = "data/processed/trimet_monthly.csv"
AGENCY = "Tri-County Metropolitan Transportation District of Oregon"
WINDOW_START = "2019-01-01"
UPT_SHEET = "UPT"
VRH_SHEET = "VRH"
CORE_MODES = ["MB", "LR"]
MODE_NAMES = {
    "MB": "Bus",
    "LR": "MAX light rail",
    "DR": "Paratransit",
    "YR": "WES",
}


def load_trimet_long(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet)
    df = df[df["Agency"].astype(str).str.contains(AGENCY, na=False)]
    assert len(df) == 8, f"Expected 8 TriMet rows, found {len(df)}"
    month_cols = [col for col in df.columns if re.fullmatch(r"\d{1,2}/\d{4}", str(col))]
    long = df.melt(
        id_vars=["Mode", "TOS"],
        value_vars=month_cols,
        var_name="month",
        value_name="value",
    )
    long["month"] = pd.to_datetime(long["month"].astype(str), format="%m/%Y")

    cleaned = pd.to_numeric(
        long["value"].astype(str).str.replace(",", "").str.strip(), errors="coerce"
    )
    assert (
        cleaned.notna().sum() == long["value"].notna().sum()
    ), "Some values could not be converted to numeric"
    long["value"] = cleaned
    return long

## Load UPT and VRH

Each sheet becomes a long table with one row per mode, type of service, month.

In [2]:
upt = load_trimet_long(PATH, UPT_SHEET)
vrh = load_trimet_long(PATH, VRH_SHEET)

## Merge UPT and VRH

Merges the UPT and VRH sheet on mode, type of service, month. Renames value columns to distinguish between UPT and VRH values

In [3]:
def merge_upt_vrh(upt_df, vrh_df):
    merged = pd.merge(
        upt_df.rename(columns={"value": "upt"}),
        vrh_df.rename(columns={"value": "vrh"}),
        on=["Mode", "TOS", "month"],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    assert (merged["_merge"] == "both").all(), "Some rows exist in only one sheet"
    return merged.drop(columns="_merge")


merged = merge_upt_vrh(upt, vrh)
print(upt.shape, vrh.shape, merged.shape)  # expect (2376, 4) (2376, 4) (2376, 5)
merged.head(10)

(2376, 4) (2376, 4) (2376, 5)


,Mode,TOS,month,upt,vrh
0,CR,PT,2002-01-01,NaN,NaN
1,CR,PT,2002-02-01,NaN,NaN
2,CR,PT,2002-03-01,NaN,NaN
3,CR,PT,2002-04-01,NaN,NaN
4,CR,PT,2002-05-01,NaN,NaN
5,CR,PT,2002-06-01,NaN,NaN
6,CR,PT,2002-07-01,NaN,NaN
7,CR,PT,2002-08-01,NaN,NaN
8,CR,PT,2002-09-01,NaN,NaN
9,CR,PT,2002-10-01,NaN,NaN


## Trim to the analysis window

- Keep only January 2019 and onward. 
- Drop rows where UPT and VRH are blank. These are inactive rows, before service began or after service ended for a particular mode, type of service. 
- Rename the columns to be lowercase. 
- Add a readable mode name. 
- Order the columns.

In [4]:
df = merged[merged["month"] >= pd.to_datetime(WINDOW_START)].copy()
df = df[df[["vrh", "upt"]].notna().any(axis=1)]

df = df.rename(columns={"Mode": "mode", "TOS": "tos"})
df["mode_name"] = df["mode"].map(MODE_NAMES)

df = df.sort_values(["mode", "tos", "month"]).reset_index(drop=True)
df = df[["month", "mode", "tos", "mode_name", "upt", "vrh"]]

LAST_MONTH = df["month"].max()

print(df.shape)  # expect (506, 6)
print("Last month:", LAST_MONTH.strftime("%Y-%m"))  # expect 2026-09
print(df.groupby(["mode", "tos"]).size())

(506, 6)
Last month: 2026-09
mode  tos
DR    PT     93
      TN     41
      TX     93
LR    DO     93
MB    DO     93
YR    PT     93
dtype: int64


## Bus and MAX share of total UPT

- Check what share of total ridership (UPT) the bus and MAX light rail make up. 
- Check the 2019 calendar year and the latest twelve months in the FTA data. 

In [5]:
# calculate what share of total ridership is from core modes (bus and light rail)
def core_share(frame):
    core_df = frame[frame["mode"].isin(CORE_MODES)]
    return core_df["upt"].sum() / frame["upt"].sum()


cal_2019 = df[df["month"].dt.year == 2019]
latest_12 = df[df["month"] > (LAST_MONTH - pd.DateOffset(months=12))]

print(f"Months in latest window: {latest_12['month'].nunique()}")  # expect 12
print(f"Bus + MAX share of UPT, calendar 2019: {core_share(cal_2019):.1%}")
print(f"Bus + MAX share of UPT, latest 12 months: {core_share(latest_12):.1%}")

Months in latest window: 12
Bus + MAX share of UPT, calendar 2019: 98.7%
Bus + MAX share of UPT, latest 12 months: 98.5%


**Bus and MAX share results:** In 2019 the bus and MAX light rail made up 98.7% of total UPT and in the last twelve months reported (ending in September 2026) they made up 98.5% of total UPT, confirming that they make up nearly all of TriMet ridership. 

## Data validation

Checks before saving:
- No duplicate keys
- No negative values
- UPT and VRH are present together in all rows
- Core modes (bus and MAX) have every month from January 2019 to the last month of data
- Zero values listed for review (any zero is a reported value, not a blank)

In [6]:
# Check key uniqueness
assert not df.duplicated(
    subset=["month", "mode", "tos"]
).any(), "Duplicate keys found in final DataFrame"

# Check for negative values
assert (
    (df[["upt", "vrh"]] >= 0).all().all()
), "Negative values found in UPT or VRH columns"

# Check for missing values in UPT and VRH columns
assert df[["upt", "vrh"]].notna().all(axis=1).all(), "Some rows have UPT or VRH missing"

# Check that core modes have every month from January 2019 to the last month (September 2026)
core_df = df[df["mode"].isin(CORE_MODES)]
expected_months = pd.date_range(start=WINDOW_START, end=LAST_MONTH, freq="MS")
for mode in CORE_MODES:
    mode_months = core_df[core_df["mode"] == mode]["month"].sort_values().unique()
    assert len(mode_months) == len(
        expected_months
    ), f"Mode {mode} does not have all expected months"
    assert (mode_months == expected_months).all(), f"Mode {mode} has unexpected months"

# Zero upt and vrh values listed for later review, but not removed from the dataset
print("\nZero values:")
print(df[(df["upt"] == 0) | (df["vrh"] == 0)])


Zero values:
         month mode tos    mode_name  upt  vrh
226 2026-09-01   DR  TX  Paratransit  0.0  0.0


## Spot check against source file

Hand check a few values: edge cases and a pandemic month. 

In [7]:
check_months = pd.to_datetime([WINDOW_START, "2020-04-01", LAST_MONTH])
df[df["mode"].isin(CORE_MODES) & df["month"].isin(check_months)]

,month,mode,tos,mode_name,upt,vrh
227,2019-01-01,LR,DO,MAX light rail,3201280.0,53964.0
242,2020-04-01,LR,DO,MAX light rail,1050980.0,49257.0
319,2026-09-01,LR,DO,MAX light rail,1880533.0,47357.0
320,2019-01-01,MB,DO,Bus,4809960.0,173620.0
335,2020-04-01,MB,DO,Bus,1526440.0,151215.0
412,2026-09-01,MB,DO,Bus,3753635.0,156209.0


**Spot check results:** The check values are consistent between the DataFrame and the source Excel file. 

## Save and export

Save the CSV and commit it so the analysis can be run without the raw Excel files. 

In [8]:
df.to_csv(OUT_PATH, index=False)

# Reload to confirm it saves and loads correctly
reloaded_df = pd.read_csv(OUT_PATH, parse_dates=["month"])
assert reloaded_df.shape == df.shape
pd.testing.assert_frame_equal(reloaded_df, df, check_dtype=False)
print(
    reloaded_df.shape,
    reloaded_df["month"].min().date(),
    reloaded_df["month"].max().date(),
)  # expect (506, 6) 2019-01-01 2026-09-01
reloaded_df.dtypes

(506, 6) 2019-01-01 2026-09-01


month        datetime64[ns]
mode                 object
tos                  object
mode_name            object
upt                 float64
vrh                 float64
dtype: object

## Conclusion

Produced `data/processed/trimet_monthly.csv`: 506 rows covering bus, MAX, paratransit, and WES, January 2019 to September 2026, with UPT and VRH side by side. Inactive rows (CR and SR) are excluded because they have no data in the window, and DR/TN has 41 months because it began in May 2023, so the 506 rows are five series with 93 months each plus 41 for DR/TN.

**Bus and MAX share of total UPT:** 98.7% in 2019 and 98.5% for Oct 2025 to Sep 2026.

**Checks:** no duplicate keys, no negative values, UPT and VRH present together, bus and MAX have all 93 months. One zero value found for paratransit (2026-09-01, DR, TX) (mode not included in analysis).

**Next:** SQL analysis in notebook 03.